# Enumerable Extensions

Runnable samples for every method in `CSharpHelperExtensions.Enumerable`.  
Run the **Setup** cell first, then any section independently.

| Section | Methods |
|---|---|
| [1. Collection Presence Shortcuts](#1-collection-presence-shortcuts) | `HasAny` · `OrEmpty` · `None` |
| [2. Materialization Helpers](#2-materialization-helpers) | `WhereNotNull` · `AsReadOnlyList` · `ToHashSetSafe` |
| [3. Sequence Transforms](#3-sequence-transforms) | `Yield` · `JoinAsString` · `WithIndex` |
| [4. Dictionary](#4-dictionary) | `ToDictionarySafe` |
| [5. Conditional Mutation](#5-conditional-mutation) | `AddIf` · `AddRangeIf` |
| [6. Conditional Concatenation](#6-conditional-concatenation) | `ConcatIf` |
| [7. Predicate Queries](#7-predicate-queries) | `None(predicate)` · `IsSingle` · `IsSingle(predicate)` · `IndexOf` |
| [8. Splitting and Chunking](#8-splitting-and-chunking) | `Partition` · `Batch` |
| [9. Min/Max Defaults](#9-minmax-defaults) | `MinByOrDefault` · `MaxByOrDefault` |
| [10. Async Projection](#10-async-projection) | `SelectAsync` · `WhenAllList` |
| [11. Existing Methods](#11-existing-methods) | `IsNullOrEmpty` · `CleanNullOrEmptyItems` · `ContainsOnly` · `AreEqual` · `ForEach` · `Reduce` |

## Setup

> **Run this cell first.** It loads the compiled library and imports the required namespaces.
>
> Build first if the DLL is missing: `dotnet build` from the repo root.

In [ ]:
#r "../src/CSharpHelperExtensions/bin/Debug/net10.0/CSharpHelperExtensions.dll"
using System.Collections.Generic;
using System.Linq;
using CSharpHelperExtensions;           // IsNullOrEmpty (string), In, IsBetween, ToJson
using CSharpHelperExtensions.Enumerable; // all EnumerableExtensions

---
## 1. Collection Presence Shortcuts

| Method | Signature | Returns |
|---|---|---|
| `HasAny` | `IEnumerable<T> → bool` | `true` when non-null and has at least one element |
| `OrEmpty` | `IEnumerable<T> → IEnumerable<T>` | original sequence, or `Empty<T>()` when null |
| `None` | `IEnumerable<T> → bool` | `true` when null or empty |

In [ ]:
// HasAny — true only when non-null and has at least one element
display(new[] { 1, 2, 3 }.HasAny());                   // True
display(new int[0].HasAny());                           // False  (empty)
display(((IEnumerable<int>)null).HasAny());             // False  (null)

In [ ]:
// OrEmpty — safe null-to-empty coalesce; non-null sequences pass through unchanged
display(((IEnumerable<int>)null).OrEmpty().Count());    // 0  (null → empty)
display(new[] { 1, 2, 3 }.OrEmpty().Count());           // 3  (unchanged)
display(new int[0].OrEmpty().Count());                  // 0  (empty stays empty)

In [ ]:
// None — true when null or empty (opposite of HasAny)
display(((IEnumerable<string>)null).None());            // True  (null)
display(new string[0].None());                          // True  (empty)
display(new[] { "a", "b" }.None());                    // False  (has elements)

---
## 2. Materialization Helpers

| Method | Signature | Returns |
|---|---|---|
| `WhereNotNull` | `IEnumerable<T?> → IEnumerable<T>` | filters out null reference elements |
| `AsReadOnlyList` | `IEnumerable<T> → IReadOnlyList<T>` | materializes to a read-only list |
| `ToHashSetSafe` | `IEnumerable<T> → HashSet<T>` | null-safe dedup to HashSet |

In [ ]:
// WhereNotNull — strips null reference elements; returns empty sequence for null source
var withNulls = new string?[] { "apple", null, "banana", null, "cherry" };
display(withNulls.WhereNotNull().ToList());              // ["apple", "banana", "cherry"]

display(((IEnumerable<string?>)null).WhereNotNull().ToList());  // []  (null source → empty)

In [ ]:
// AsReadOnlyList — materializes a lazy sequence into an IReadOnlyList<T>
IEnumerable<int> lazy = Enumerable.Range(1, 5);
IReadOnlyList<int> list = lazy.AsReadOnlyList();
display(list);                                           // [1, 2, 3, 4, 5]
display(list.Count);                                     // 5

display(((IEnumerable<int>)null).AsReadOnlyList().Count); // 0  (null → empty list)

In [ ]:
// ToHashSetSafe — deduplicates; returns empty HashSet for null source (no exception)
var dupes = new[] { 1, 2, 2, 3, 3, 3 };
display(dupes.ToHashSetSafe());                          // {1, 2, 3}
display(dupes.ToHashSetSafe().Count);                   // 3

display(((IEnumerable<int>)null).ToHashSetSafe().Count); // 0  (null → empty set)

---
## 3. Sequence Transforms

| Method | Signature | Returns |
|---|---|---|
| `Yield` | `T → IEnumerable<T>` | wraps a single value into a one-element sequence |
| `JoinAsString` | `IEnumerable<T>, string → string` | joins elements to a string with a separator |
| `WithIndex` | `IEnumerable<T> → IEnumerable<(int Index, T Item)>` | pairs each element with its zero-based index |

In [ ]:
// Yield — wraps a single value into an IEnumerable<T>
display(42.Yield().ToList());                            // [42]
display("hello".Yield().ToList());                      // ["hello"]

// Useful for concatenating a single item onto an existing sequence
var existing = new[] { 1, 2, 3 };
display(existing.Concat(99.Yield()).ToList());           // [1, 2, 3, 99]

In [ ]:
// JoinAsString — fluent string.Join with separator as argument
display(new[] { "one", "two", "three" }.JoinAsString(", "));  // "one, two, three"
display(new[] { 1, 2, 3 }.JoinAsString(" | "));               // "1 | 2 | 3"
display(new[] { "a", "b" }.JoinAsString(""));                 // "ab"  (no separator)
display(((IEnumerable<string>)null).JoinAsString(", "));       // ""  (null-safe)

In [ ]:
// WithIndex — projects (Index, Item) tuples; useful for numbered loops without a counter variable
var fruits = new[] { "apple", "banana", "cherry" };
foreach (var (index, item) in fruits.WithIndex())
    display($"{index}: {item}");               // "0: apple", "1: banana", "2: cherry"

display(((IEnumerable<string>)null).WithIndex().ToList()); // []  (null-safe)

---
## 4. Dictionary

| Method | Signature | Notes |
|---|---|---|
| `ToDictionarySafe` | `IEnumerable<TSource>, keySelector, valueSelector → Dictionary<TKey, TValue>` | null-safe; last value wins on duplicate keys |

In [ ]:
// ToDictionarySafe — null-safe conversion to Dictionary; duplicate keys don't throw
var pairs = new[] { ("a", 1), ("b", 2), ("c", 3) };
var dict = pairs.ToDictionarySafe(x => x.Item1, x => x.Item2);
display(dict);                                           // {a: 1, b: 2, c: 3}

// Last-value-wins on duplicate keys (unlike standard ToDictionary which throws)
var withDupes = new[] { ("a", 1), ("b", 2), ("a", 99) };
var dictDupes = withDupes.ToDictionarySafe(x => x.Item1, x => x.Item2);
display(dictDupes["a"]);                                 // 99  (last value wins)
display(dictDupes["b"]);                                 // 2

// Null source returns empty dictionary (no NullReferenceException)
IEnumerable<(string, int)> nullSource = null;
display(nullSource.ToDictionarySafe(x => x.Item1, x => x.Item2).Count); // 0

---
## 5. Conditional Mutation

Both methods operate on `IList<T>` and return the **same list instance** for fluent chaining.

| Method | Signature | Notes |
|---|---|---|
| `AddIf` | `IList<T>, bool, T → IList<T>` | adds single item when condition is true |
| `AddRangeIf` | `IList<T>, bool, IEnumerable<T> → IList<T>` | adds multiple items when condition is true |

In [ ]:
// AddIf — adds item only when condition is true; returns same list for chaining
var list = new List<int> { 1, 2 };
list.AddIf(true, 3);                                     // adds 3
list.AddIf(false, 4);                                    // skipped
display(list);                                           // [1, 2, 3]

// Fluent chaining
bool includeBonus = true;
bool includeExtra = false;
var scores = new List<int> { 10, 20 }
    .AddIf(includeBonus, 5)
    .AddIf(includeExtra, 100);
display(scores);                                         // [10, 20, 5]

In [ ]:
// AddRangeIf — adds a range of items only when condition is true; returns same list for chaining
var items = new List<string> { "base" };
items.AddRangeIf(true, new[] { "extra1", "extra2" });   // adds both
items.AddRangeIf(false, new[] { "skip1", "skip2" });    // skipped
display(items);                                          // ["base", "extra1", "extra2"]

// null items parameter is treated as empty (no exception)
var safe = new List<int> { 1 };
safe.AddRangeIf(true, (IEnumerable<int>)null);
display(safe);                                           // [1]  (unchanged)

---
## 6. Conditional Concatenation

| Method | Signature | Notes |
|---|---|---|
| `ConcatIf` | `IEnumerable<T>, bool, IEnumerable<T> → IEnumerable<T>` | concatenates second sequence only when condition is true |

In [ ]:
// ConcatIf — returns source + other when condition is true, otherwise just source
display(new[] { 1, 2 }.ConcatIf(true, new[] { 3, 4 }).ToList());   // [1, 2, 3, 4]
display(new[] { 1, 2 }.ConcatIf(false, new[] { 3, 4 }).ToList());  // [1, 2]  (not concatenated)

In [ ]:
// Null-safe on both sides
display(((IEnumerable<int>)null).ConcatIf(true, new[] { 1, 2 }).ToList());   // [1, 2]  (null source → empty)
display(((IEnumerable<int>)null).ConcatIf(false, new[] { 1, 2 }).ToList());  // []  (condition false)
display(new[] { 1, 2 }.ConcatIf(true, (IEnumerable<int>)null).ToList());     // [1, 2]  (null other → empty)

In [ ]:
// Practical use: build a filter list conditionally
bool includeArchived = false;
bool includeDeleted = true;
var statuses = new[] { "active", "pending" }
    .ConcatIf(includeArchived, new[] { "archived" })
    .ConcatIf(includeDeleted, new[] { "deleted" })
    .ToList();
display(statuses);                                       // ["active", "pending", "deleted"]

---
## 7. Predicate Queries

| Method | Signature | Returns |
|---|---|---|
| `None(predicate)` | `IEnumerable<T>, Func<T,bool> → bool` | `true` when no element matches, or source is null |
| `IsSingle` | `IEnumerable<T> → bool` | `true` when exactly one element exists |
| `IsSingle(predicate)` | `IEnumerable<T>, Func<T,bool> → bool` | `true` when exactly one element matches |
| `IndexOf` | `IEnumerable<T>, Func<T,bool> → int` | first matching index, or `-1` |

In [ ]:
// None(predicate) — true when no element satisfies the predicate
display(new[] { 1, 2, 3 }.None(x => x > 10));          // True  (none above 10)
display(new[] { 1, 2, 3 }.None(x => x > 2));           // False  (3 is above 2)
display(((IEnumerable<int>)null).None(x => x > 0));     // True  (null source)

In [ ]:
// IsSingle — true when the sequence contains exactly one element
display(new[] { 42 }.IsSingle());                        // True  (one element)
display(new[] { 1, 2 }.IsSingle());                      // False  (two elements)
display(new int[0].IsSingle());                          // False  (empty)
display(((IEnumerable<int>)null).IsSingle());             // False  (null)

In [ ]:
// IsSingle(predicate) — true when exactly one element matches the predicate
display(new[] { 1, 5, 2 }.IsSingle(x => x > 3));        // True  (only 5 > 3)
display(new[] { 1, 5, 6 }.IsSingle(x => x > 3));        // False  (both 5 and 6 > 3)
display(new[] { 1, 2, 3 }.IsSingle(x => x > 10));       // False  (none matches)
display(((IEnumerable<int>)null).IsSingle(x => x > 0));  // False  (null source)

In [ ]:
// IndexOf — returns zero-based index of the first matching element, or -1 if not found
var names = new[] { "Alice", "Bob", "Charlie", "Dave" };
display(names.IndexOf(n => n.StartsWith("C")));          // 2  ("Charlie")
display(names.IndexOf(n => n.StartsWith("Z")));          // -1  (no match)
display(names.IndexOf(n => n.Length > 3));               // 0  ("Alice" is first match)

display(((IEnumerable<string>)null).IndexOf(n => true)); // -1  (null source)

---
## 8. Splitting and Chunking

| Method | Signature | Returns |
|---|---|---|
| `Partition` | `IEnumerable<T>, Func<T,bool> → (Matched, Remaining)` | splits into two lists by predicate |
| `Batch` | `IEnumerable<T>, int → IEnumerable<IReadOnlyList<T>>` | splits into chunks of at most `size` elements |

In [ ]:
// Partition — splits sequence into (Matched, Remaining) based on a predicate
var numbers = new[] { 1, 2, 3, 4, 5, 6 };
var (evens, odds) = numbers.Partition(x => x % 2 == 0);
display(evens);                                          // [2, 4, 6]
display(odds);                                           // [1, 3, 5]

In [ ]:
// Partition — practical use: separate valid from invalid items
var inputs = new[] { "alice@example.com", "not-an-email", "bob@test.org", "" };
var (valid, invalid) = inputs.Partition(s => s.Contains('@'));
display(valid);                                          // ["alice@example.com", "bob@test.org"]
display(invalid);                                        // ["not-an-email", ""]

// Null source returns two empty lists
var (m, r) = ((IEnumerable<int>)null).Partition(x => x > 0);
display(m.Count);                                        // 0
display(r.Count);                                        // 0

In [ ]:
// Batch — splits sequence into chunks of a given size; last chunk may be smaller
var items = Enumerable.Range(1, 10).ToList();
var batches = items.Batch(3).ToList();
display(batches.Count);                                  // 4 batches
display(batches[0]);                                     // [1, 2, 3]
display(batches[1]);                                     // [4, 5, 6]
display(batches[2]);                                     // [7, 8, 9]
display(batches[3]);                                     // [10]  (last chunk smaller)

In [ ]:
// Batch — null source returns empty sequence
display(((IEnumerable<int>)null).Batch(5).ToList().Count); // 0

// Batch size larger than sequence — single chunk
display(new[] { 1, 2 }.Batch(10).ToList().Count);        // 1  (one chunk with both elements)

---
## 9. Min/Max Defaults

Null-safe wrappers around LINQ's `MinBy`/`MaxBy` that return `default(T)` instead of throwing on null or empty sequences.

| Method | Signature | Notes |
|---|---|---|
| `MinByOrDefault` | `IEnumerable<T>, Func<T,TKey> → T?` | smallest by key, or `default` when null/empty |
| `MaxByOrDefault` | `IEnumerable<T>, Func<T,TKey> → T?` | largest by key, or `default` when null/empty |

In [ ]:
// MinByOrDefault — returns element with smallest key value
display(new[] { 3, 1, 4, 1, 5, 9 }.MinByOrDefault(x => x));    // 1
display(((IEnumerable<int>)null).MinByOrDefault(x => x));       // 0  (default for int)
display(Enumerable.Empty<string>().MinByOrDefault(x => x));     // null  (default for string)

In [ ]:
// MaxByOrDefault — returns element with largest key value
display(new[] { 3, 1, 4, 1, 5, 9 }.MaxByOrDefault(x => x));    // 9
display(((IEnumerable<int>)null).MaxByOrDefault(x => x));       // 0  (default for int)
display(Enumerable.Empty<string>().MaxByOrDefault(x => x));     // null  (default for string)

In [ ]:
// Min/Max on objects by property — common real-world use
var people = new[]
{
    (Name: "Alice", Age: 30),
    (Name: "Bob",   Age: 25),
    (Name: "Carol", Age: 35),
};
display(people.MinByOrDefault(p => p.Age));             // (Bob, 25)
display(people.MaxByOrDefault(p => p.Age));             // (Carol, 35)

---
## 10. Async Projection

| Method | Signature | Notes |
|---|---|---|
| `SelectAsync` | `IEnumerable<T>, Func<T,Task<TResult>>, int? → Task<IReadOnlyList<TResult>>` | async map; optional concurrency cap via `maxParallel` |
| `WhenAllList` | `IEnumerable<Task<T>> → Task<IReadOnlyList<T>>` | awaits all tasks, returns results as read-only list |

In [ ]:
// SelectAsync — async projection of each element; all run concurrently by default
async Task<string> FetchLabel(int id)
{
    await Task.Delay(1);   // simulate async work
    return $"item-{id}";
}

var results = await new[] { 1, 2, 3, 4, 5 }.SelectAsync(FetchLabel);
display(results);                                        // ["item-1", "item-2", "item-3", "item-4", "item-5"]

In [ ]:
// SelectAsync with maxParallel — limits concurrency (useful for rate-limited APIs)
var throttled = await new[] { 1, 2, 3, 4, 5 }.SelectAsync(FetchLabel, maxParallel: 2);
display(throttled);                                      // ["item-1", "item-2", "item-3", "item-4", "item-5"]

// Null source returns empty list without throwing
var empty = await ((IEnumerable<int>)null).SelectAsync(FetchLabel);
display(empty.Count);                                    // 0

In [ ]:
// WhenAllList — awaits a sequence of tasks and collects results into IReadOnlyList<T>
var tasks = new[] { 10, 20, 30 }.Select(async n =>
{
    await Task.Delay(1);
    return n * 2;
});

var doubled = await tasks.WhenAllList();
display(doubled);                                        // [20, 40, 60]

// Null source returns empty list
var nullResult = await ((IEnumerable<Task<int>>)null).WhenAllList();
display(nullResult.Count);                               // 0

---
## 11. Existing Methods

These methods were present before the new additions. Shown here for completeness.

| Method | Notes |
|---|---|
| `IsNullOrEmpty` | `true` when null, empty, or all-null elements |
| `CleanNullOrEmptyItems` | removes null (and for strings: empty/whitespace) elements |
| `ContainsOnly` | `true` when sequence contains exactly the specified items (any order) |
| `AreEqual` | order-sensitive or order-insensitive sequence equality |
| `ForEach` | side-effectful iteration; returns original sequence |
| `Reduce` | fold/accumulate to a single value |

In [ ]:
// IsNullOrEmpty — true for null, empty, or all-null sequences
display(((IEnumerable<int>)null).IsNullOrEmpty());       // True
display(new List<string>().IsNullOrEmpty());             // True
display(new string[] { null, null }.IsNullOrEmpty());    // True  (all-null elements)
display(new[] { 1, 2, 3 }.IsNullOrEmpty());             // False

In [ ]:
// CleanNullOrEmptyItems — removes nulls; for string sequences also removes empty/whitespace
display(new[] { "hello", null, "", "  ", "world" }.CleanNullOrEmptyItems().ToList());
// ["hello", "world"]

display(new int?[] { 1, null, 2, null, 3 }.CleanNullOrEmptyItems().ToList());
// [1, 2, 3]

In [ ]:
// ContainsOnly — true when sequence contains exactly these items (order-insensitive)
display(new[] { 1, 2, 3 }.ContainsOnly(3, 1, 2));       // True  (same items, different order)
display(new[] { 1, 2, 3 }.ContainsOnly(1, 2));          // False  (extra element in source)
display(new[] { 1, 2 }.ContainsOnly(1, 2, 3));          // False  (missing element)

In [ ]:
// AreEqual — sequence equality with optional order sensitivity
display(new[] { 1, 2, 3 }.AreEqual(new[] { 3, 1, 2 }));                      // True  (NoOrder default)
display(new[] { 1, 2, 3 }.AreEqual(new[] { 3, 1, 2 }, Compare.InOrder));     // False  (order differs)
display(new[] { 1, 2, 3 }.AreEqual(new[] { 1, 2, 3 }, Compare.InOrder));     // True
display(((IEnumerable<int>)null).AreEqual(null));                              // True  (both null)

In [ ]:
// ForEach — side-effectful iteration; returns original sequence for chaining
var log = new List<string>();
new[] { "a", "b", "c" }
    .ForEach(item => log.Add(item.ToUpper()))
    .ForEach(item => display(item));                     // prints "a", "b", "c"
display(log);                                            // ["A", "B", "C"]

In [ ]:
// Reduce — fold to single accumulated value
// Sum integers
display(new[] { 1, 2, 3, 4 }.Reduce((item, acc) => acc + item, initialValue: 0));   // 10

// Build comma-separated string
display(new[] { "a", "b", "c" }.Reduce(
    (item, acc) => acc == "" ? item : acc + ", " + item, ""));                       // "a, b, c"

// Reduce with index — build numbered list
display(new[] { "apple", "banana", "cherry" }.Reduce(
    (item, acc, index) => acc + $"{index}: {item}\n", ""));
// "0: apple\n1: banana\n2: cherry\n"